# Step 13.3 — Split Channels into Heads：怎样把 `(B,T,C)` 组织成 `(B,T,H,D)`？

Step 13.2 已经得到 full-width Q/K/V：

$$
Q,K,V\in\mathbb R^{B\times T\times C}.
$$

现在 multiple heads 还没有真正出现在 tensor shape 里。

下一步就是把最后一个 feature axis `C` 拆成：

$$
\boxed{C=H\times D}
$$

其中本节：

```text
H = number of heads（head 数量）
D = head dimension（每个 head 的 feature width）
```

注意：这里的 `H` 是 Multi-Head Attention 的 **head count**，不是前面 RNN 章节里曾用过的 hidden width 记号。

然后把：

$$
(B,T,C)
\rightarrow
\boxed{(B,T,H,D)}.
$$

本节只做 channel-to-head organization；不进入 Step 13.4 的 `(B,T,H,D) -> (B,H,T,D)` transpose，也不计算 batched multi-head attention。


## 1. 为什么必须满足 `C = H × D`？

对每一个 token position，Q/K/V 当前都有一个长度为 `C` 的 feature vector：

$$
q_{b,t,:}\in\mathbb R^C.
$$

如果要把这 `C` 个 channels 平均组织成 `H` 个 heads，那么每个 head 必须拿到相同数量的 channels：

$$
\boxed{D=\frac{C}{H}}.
$$

因此最简单的 GPT-style equal-width heads 要求：

$$
\boxed{C\bmod H=0}.
$$

例如：

```text
C = 12, H = 3  -> D = 4   ✓
C = 12, H = 4  -> D = 3   ✓
C = 12, H = 5  -> 不是整数 ✗
```

这不是 Attention 数学要求“永远只能等宽 heads”，而是当前标准等宽 Multi-Head tensor organization 的要求。


## 2. Reshape 的精确含义：把一个 Channel Index c 写成 `(h,d)`

原来最后一个 axis 的 channel index：

$$
c=0,1,\ldots,C-1.
$$

如果：

$$
C=H\times D,
$$

那么每个 `c` 都可以唯一写成：

$$
\boxed{c=hD+d}
$$

其中：

$$
h\in\{0,\ldots,H-1\},
\qquad
d\in\{0,\ldots,D-1\}.
$$

所以 reshape 后可以精确理解为：

$$
\boxed{
Q_{heads}[b,t,h,d]
=
Q[b,t,hD+d]
}
$$

K、V 完全相同。

这一步：

- 不产生新的 learned parameters；
- 不做新的 Linear projection；
- 不改变元素总数；
- 不把某个 channel 数值做加法或平均。

它只是把原来的一个 feature axis `(C)` **逻辑上重新解释**成两个 axes `(H,D)`。

至于底层实现是否一定 zero-copy，是 runtime / memory-layout 细节；本节只讨论 tensor semantics。


## 3. Tiny Example：`C=6, H=3, D=2`

取一个 token 的 Query vector：

$$
q=[10,11,20,21,30,31].
$$

这里：

```text
C = 6
H = 3 heads
D = 2 channels per head
```

reshape 成 `(H,D)` 后：

$$
q_{heads}
=
\begin{bmatrix}
10&11\\
20&21\\
30&31
\end{bmatrix}.
$$

所以：

```text
head 0 -> [10,11]
head 1 -> [20,21]
head 2 -> [30,31]
```

注意这里不是说：

```text
head 0 天生负责某种语义
head 1 天生负责另一种语义
```

这些 channels 是前面 learned Q/K/V projection 的输出。训练会决定这些 channel directions 有什么用途；reshape 只是把它们按固定 channel groups 组织成 heads。


## 4. 加回 Sequence：`(T,C) -> (T,H,D)`

设：

$$
T=2,\quad C=6,\quad H=3,\quad D=2.
$$

一个 Query tensor：

$$
Q=
\begin{bmatrix}
0&1&2&3&4&5\\
6&7&8&9&10&11
\end{bmatrix}
\in\mathbb R^{2\times6}.
$$

reshape 后：

$$
Q_{heads}\in\mathbb R^{2\times3\times2}.
$$

逻辑内容是：

```text
token 0:
  head 0 -> [0,1]
  head 1 -> [2,3]
  head 2 -> [4,5]

token 1:
  head 0 -> [6,7]
  head 1 -> [8,9]
  head 2 -> [10,11]
```

所以 T axis 仍然保留：每一个 token position 都拥有 H 个 head-specific channel blocks。


In [ ]:
import mlx.core as mx

B, T, C = 1, 2, 6
H = 3

assert C % H == 0
D = C // H

Q = mx.reshape(
    mx.arange(B * T * C),
    (B, T, C),
)

Q_heads = mx.reshape(Q, (B, T, H, D))

print('C:', C)
print('H:', H)
print('D:', D)
print('Q shape:', Q.shape)
print('Q:')
print(Q)
print('Q_heads shape:', Q_heads.shape)
print('Q_heads:')
print(Q_heads)

# Expected logical grouping when executed:
# token 0 -> heads [0,1], [2,3], [4,5]
# token 1 -> heads [6,7], [8,9], [10,11]
# This master notebook remains unexecuted.


## 5. 对 Q、K、V 都做同样的 Head Split

承接 Step 13.2：

```text
qkv : (B,T,3C)
↓ split Q/K/V
q : (B,T,C)
k : (B,T,C)
v : (B,T,C)
```

现在分别 reshape：

$$
Q:(B,T,C)
\rightarrow
Q_h:(B,T,H,D),
$$

$$
K:(B,T,C)
\rightarrow
K_h:(B,T,H,D),
$$

$$
V:(B,T,C)
\rightarrow
V_h:(B,T,H,D).
$$

每一个 tensor 的元素数量保持：

$$
B\times T\times C
=
B\times T\times H\times D.
$$

因为：

$$
C=H\times D.
$$

所以这是合法 reshape。


In [ ]:
# Assume q, k, v already came from Step 13.2 and each has shape (B,T,C).
q = mx.zeros((B, T, C))
k = mx.zeros((B, T, C))
v = mx.zeros((B, T, C))

q_heads = mx.reshape(q, (B, T, H, D))
k_heads = mx.reshape(k, (B, T, H, D))
v_heads = mx.reshape(v, (B, T, H, D))

print('q_heads:', q_heads.shape)
print('k_heads:', k_heads.shape)
print('v_heads:', v_heads.shape)

# No head-axis transpose yet.
# Shapes intentionally remain (B,T,H,D) in Step 13.3.


## 6. 这里的 Head 不是 H 个单独 Python Modules

一个很容易形成的误解是：

```text
H heads
=
H 个独立 nn.Module 对象逐个循环计算
```

这不是当前 GPT-style tensor organization 的核心。

13.2 已经用一个 fused projection 得到 full-width：

$$
Q,K,V:(B,T,C).
$$

13.3 只是把 `C` 解释成：

$$
H\times D.
$$

所以 heads 首先体现为 **tensor axis organization**：

$$
(B,T,H,D).
$$

不同 head 的 channels 对应 fused projection 输出中的不同 channel blocks，因此这些 blocks 的 learned projection parameters 仍然可以不同。

这种 tensorized organization 为后续一次性做 batched multi-head computation 做准备。

但“怎样把 H 放到适合矩阵乘法的位置”属于 Step 13.4，本节不交换 axes。


## 7. Reshape 会增加 Parameters 吗？不会

Step 13.2 的 fused projection 已经拥有：

$$
3C^2
$$

个 weights（bias=False 情况）。

执行：

$$
(B,T,C)
\rightarrow
(B,T,H,D)
$$

只改变 tensor shape / logical indexing，不创建新的 learned matrix。

因此：

$$
\boxed{
\text{parameter count before reshape}
=
\text{parameter count after reshape}
}
$$

Multi-Head 的 head-specific learned freedom 已经编码在 Q/K/V projection 输出 channels 中；reshape 只是把这些 channels 显式组织成 head groups。


## 8. Step 13.3 材料边界

- 在本节记号中 `H` 是 number of heads，`D` 是 head dimension；标准等宽 heads 满足 `C=H×D`，因此 `C % H == 0`。
- `(B,T,C)->(B,T,H,D)` 是 channel-to-head reshape：不增加、删除或聚合元素，也不新增 learned parameters。
- 精确索引关系是 `Q_heads[b,t,h,d] = Q[b,t,hD+d]`；K/V 同理。
- head grouping 来自 learned Q/K/V output channels 的固定分块，不应给这些 channel blocks 预设固定人类语义。
- `mx.reshape(a, shape)` 在 MLX 0.32.2 中保持元素总数；本节用它把 Q/K/V 分别组织成 `(B,T,H,D)`。
- 当前 heads 只是显式成为一个 tensor axis；本节没有把 H 个 heads 实现成 H 个 Python module loops。
- 本轮只准备 Step 13.3；不进入 Step 13.4，不做 `(B,T,H,D)->(B,H,T,D)` transpose，不计算 batched `QK^T`，也不讨论 GPU core 对应关系。
